# BIO-MIMIC X — Kaggle HAND2 genome scan\nCPU-first. No paid infrastructure. Streaming FASTA. A complete assembly and validation are required before uniqueness claims.

In [ ]:
# BIO-MIMIC X: bounded, auditable HAND2 31-mer scan (Kaggle CPU)
import gzip, hashlib, json, pathlib, urllib.request
K=31
QUERY_ACCESSION="CM010938.2"; START=884794726; END=884799066
MODERN="NC_090937.1"; MODERN_START=910028725; MODERN_END=910032936
OUT=pathlib.Path("/kaggle/working" if pathlib.Path("/kaggle/working").exists() else ".")
def efetch(acc,start,end):
    from urllib.parse import urlencode
    url="https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?"+urlencode(dict(db="nuccore",id=acc,seq_start=start,seq_stop=end,rettype="fasta",retmode="text"))
    with urllib.request.urlopen(urllib.request.Request(url,headers={"User-Agent":"BIO-MIMIC-X/1.0"}),timeout=60) as f: lines=f.read().decode().splitlines()
    assert lines and lines[0].startswith(">"+acc+":"+str(start)+"-"+str(end)),lines[:1]
    seq="".join(lines[1:]).upper(); assert len(seq)==end-start+1
    return seq
def rc(s): return s.translate(str.maketrans("ACGTN","TGCAN"))[::-1]
def kmers(s,k=K): return {s[i:i+k] for i in range(len(s)-k+1) if "N" not in s[i:i+k]}
def scan_fasta(path, query, k=K):
    # FASTA streamed: bounded memory; supports .fa, .fasta and .gz; no whole-genome download assumed
    op=gzip.open if str(path).endswith(".gz") else open
    q=kmers(query,k); hits={}; contig=None; tail=""; offset=0; total=0
    with op(path,"rt") as fh:
        for line in fh:
            if line.startswith(">"):
                contig=line[1:].split()[0]; tail=""; offset=0; continue
            s=line.strip().upper(); chunk=tail+s; base=offset-len(tail)
            for i in range(max(0,len(chunk)-k+1)):
                word=chunk[i:i+k]
                if word in q:
                    p=base+i+1; hits.setdefault(contig,[]).append(p)
            tail=chunk[-(k-1):]; offset+=len(s); total+=len(s)
    return {"scanned_bases":total,"query_distinct_31mers":len(q),"contigs":{c:{"match_occurrences":len(v),"first_positions_1based":v[:20]} for c,v in hits.items()}}
# Sanity tests: positive, reverse-complement and negative control
assert kmers("ACGT"*10)==kmers("ACGT"*10)
assert rc(rc("ACGTN"))=="ACGTN"
assert len(kmers("A"*40))==1
# Set FASTA_PATH to a legally accessible, version-pinned genome FASTA in Kaggle input.
# Do NOT mark genome-wide uniqueness unless the FASTA is complete and accession/hash verified.
FASTA_PATH=None
if FASTA_PATH:
    query=efetch(QUERY_ACCESSION,START,END)
    result=scan_fasta(FASTA_PATH,query)
    result.update({"query_accession":QUERY_ACCESSION,"query_interval":[START,END],"fasta_path":str(FASTA_PATH),"status":"PARTIAL_UNTIL_ASSEMBLY_COMPLETENESS_AND_RANKING_VALIDATED"})
    (OUT/"hand2_kaggle_scan.json").write_text(json.dumps(result,indent=2))
    print(result["scanned_bases"],len(result["contigs"]))
else:
    print("PASS local sanity tests; UNVERIFIED genome scan: set FASTA_PATH to a verified assembly input")
